# Redrob AI Ranker — Sandbox Demo

**India Runs Data & AI Challenge | Submission by AlfaizKureshi**

---

This notebook runs the full ranking pipeline end-to-end on a 50-candidate sample.

| What | Detail |
|---|---|
| Pipeline | Hard disqualify → 11-signal base score → behavioral multiplier → penalties |
| Runtime | ~5–15 seconds on this sample (vs ~25s on the full 100K dataset) |
| GPU | Not required — CPU only |
| Network | No external API calls during ranking |
| Output | Ranked CSV with candidate_id, rank, score, reasoning |

**To reproduce: Runtime → Run All (Ctrl+F9)**

In [ ]:
# ── Cell 1: Clone the repo ──────────────────────────────────────────────────
# Always return to /content first so this cell is safe to re-run.
# Without this, %cd redrob-ai-ranker stacks on each run and breaks paths.
%cd /content
!rm -rf redrob-ai-ranker
!git clone https://github.com/Alfaiz777/redrob-ai-ranker.git --quiet
%cd redrob-ai-ranker
print('[OK] Repository cloned — working directory: redrob-ai-ranker')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
# The core ranking engine is pure Python stdlib (json, re, multiprocessing, csv).
# pandas is only added here to display results as a readable table in the notebook.
# sentence-transformers and faiss-cpu (in requirements.txt) are for the optional
# FAISS retrieval pipeline — not needed for the standard ranking step.
!pip install pandas -q
print('[OK] Dependencies ready')

In [ ]:
# ── Cell 3: Prepare sample input ────────────────────────────────────────────
# The ranking pipeline reads JSONL (one JSON object per line).
# The repo ships sample_candidates.json (JSON array) for human readability.
# This one-time conversion makes it compatible with run.py.
import json

with open('data/sample_candidates.json', 'r', encoding='utf-8') as f:
    candidates = json.load(f)

with open('data/sample_candidates.jsonl', 'w', encoding='utf-8') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

print(f'[OK] {len(candidates)} candidates written to data/sample_candidates.jsonl')
print(f'     First candidate: {candidates[0]["candidate_id"]}')

In [ ]:
# ── Cell 4: Run the full ranking pipeline ───────────────────────────────────
# Exact same command as README.md.
# Runs: disqualify -> 11-signal score -> behavioral multiplier -> penalties -> CSV
# No GPU. No network calls. Pure CPU.
!python run.py --input data/sample_candidates.jsonl --out sample_submission.csv

In [ ]:
# ── Cell 5: Verify and display ranked results ────────────────────────────────
import pandas as pd

df = pd.read_csv('sample_submission.csv')

# Spec checks
scores_ok = all(df['score'].iloc[i] >= df['score'].iloc[i+1] for i in range(len(df)-1))
ranks_ok  = sorted(df['rank'].tolist()) == list(range(1, len(df)+1))
ids_ok    = df['candidate_id'].nunique() == len(df)

print(f'Candidates ranked      : {len(df)}')
print(f'Score range            : {df["score"].max():.4f}  (rank 1)  to  {df["score"].min():.4f}  (last rank)')
print(f'Scores non-increasing  : {scores_ok}')
print(f'Ranks 1-N each once    : {ranks_ok}')
print(f'No duplicate IDs       : {ids_ok}')
print()
pd.set_option('display.max_colwidth', 110)
display(df[['rank', 'candidate_id', 'score', 'reasoning']])

In [ ]:
# ── Cell 6: Download the CSV ─────────────────────────────────────────────────
# Triggers a file download in Colab. Skip if running locally.
try:
    from google.colab import files
    files.download('sample_submission.csv')
    print('[OK] sample_submission.csv downloaded')
except ImportError:
    print('[INFO] Not in Colab — file is at: redrob-ai-ranker/sample_submission.csv')